# DeepWiki Infrastructure Deployment

This notebook deploys all DeepWiki infrastructure resources to Azure.

## Resources Deployed
1. **User Managed Identity** — Authentication for all Azure services
2. **Azure Storage Account** — Blob storage for wiki cache and data
3. **Azure OpenAI** — LLM and embedding models
4. **Application Insights + Log Analytics** — Monitoring and diagnostics
5. **Azure App Service** — Container hosting (Linux, with network restrictions)
6. **Role Assignments** — RBAC permissions for managed identity
7. **Network Security Perimeter (NSP)** — Optional network security
8. **Azure Key Vault** — Secret storage for AML workspace (RBAC-enabled, NSP-protected)
9. **Azure Machine Learning Workspace** — Linked to existing storage, OpenAI, App Insights, Key Vault
10. **Azure AI Search Service** — Linked to existing storage account, with data source configured

## Prerequisites
- Azure CLI authenticated (`az login`)
- Python 3.10+ with required packages
- Configuration completed in `config.py`

## Deployment Order
Run cells in sequence — each block depends on previous deployments.


## Block 1: Setup & Configuration

Load configuration and authenticate with Azure.

In [ ]:
# Import required libraries
import json
import importlib
from pathlib import Path
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.resource.resources.models import ResourceGroup

# Import deployment utilities (includes parameter template resolver and resource checks)
# Reload to ensure latest changes are picked up
import utils
importlib.reload(utils)
from utils import deploy_arm, load_config, get_parameters_for_deployment, check_and_prepare_deployment

print("✓ Libraries imported successfully")

In [ ]:
# Load configuration from config.py using the utility function
config = load_config("config.py")

print("✓ Configuration loaded successfully")
print(f"  - Subscription ID: {config['subscription_id']}")
print(f"  - Resource Group: {config['resource_group']}")
print(f"  - Location: {config['location']}")
print(f"  - Managed Identity: {config['dri_copilot_identity_name']}")
print(f"  - Resource Tags: {config['resource_tags']}")

In [ ]:
# Authenticate with Azure using Azure CLI credentials
credential = AzureCliCredential()

# Test authentication
credential.get_token("https://management.azure.com/.default")

# Initialize resource management client
resource_client = ResourceManagementClient(credential, config['subscription_id'])

print("✓ Azure authentication successful")

## Block 2: Create Resource Group

Create or verify the Azure Resource Group for DeepWiki resources.

In [ ]:
# Create Resource Group
resource_group_name = config['resource_group']
location = config['location']

print(f"Creating resource group: {resource_group_name}")
print(f"Location: {location}")

try:
    # Check if resource group exists
    existing_rg = resource_client.resource_groups.get(resource_group_name)
    print(f"✓ Resource group '{resource_group_name}' already exists")
    print(f"  - Location: {existing_rg.location}")
    print(f"  - Provisioning State: {existing_rg.properties.provisioning_state}")
except Exception:
    # Create resource group
    print(f"Creating resource group '{resource_group_name}'...")
    resource_group_params = ResourceGroup(location=location, tags=config.get('resource_tags', {}))
    result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        resource_group_params
    )
    print(f"✓ Resource group created successfully")
    print(f"  - Location: {result.location}")
    print(f"  - Provisioning State: {result.properties.provisioning_state}")

## Block 3: Deploy User Managed Identity

Deploy the User-Assigned Managed Identity that will be used by all services.

In [ ]:
print("=" * 60)
print("DEPLOYING: User Managed Identity")
print("=" * 60)
print(f"Identity Name: {config['dri_copilot_identity_name']}")
print(f"Location: {config['location']}")

# Check if resource exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['dri_copilot_identity_name'],
    resource_type="managed_identity",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    managed_identity_params = get_parameters_for_deployment("UMI", config)
    
    # Deploy
    managed_identity_outputs = deploy_arm(
        template_file_path="templates/UMI.Template.json",
        deployment_name="deepwiki-managed-identity",
        parameters=managed_identity_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    # Store outputs for later use
    managed_identity_principal_id = managed_identity_outputs.get('managedIdentityPrincipalId', {}).get('value', '')
    managed_identity_client_id = managed_identity_outputs.get('managedIdentityClientId', {}).get('value', '')
    
    print(f"\n✓ Managed Identity deployed successfully")
    print(f"  - Principal ID: {managed_identity_principal_id}")
    print(f"  - Client ID: {managed_identity_client_id}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 4: Deploy Azure Storage Account

Deploy the Storage Account for blob storage (wiki cache, embeddings, etc.).

In [ ]:
print("=" * 60)
print("DEPLOYING: Azure Storage Account")
print("=" * 60)
print(f"Storage Account: {config['storage_account_name']}")
print(f"Location: {config['location']}")

# Check if resource exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['storage_account_name'],
    resource_type="storage",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    storage_params = get_parameters_for_deployment("STORAGE", config)
    
    # Deploy
    storage_outputs = deploy_arm(
        template_file_path="templates/STORAGE.Template.json",
        deployment_name="deepwiki-storage",
        parameters=storage_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    print(f"\n✓ Storage Account deployed successfully")
    if storage_outputs:
        print(f"  - Blob Endpoint: {storage_outputs.get('primaryBlobEndpoint', {}).get('value', 'N/A')}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 5: Deploy Azure OpenAI Service

Deploy Azure OpenAI service with model deployments.

**Note:** Run this block only if `is_creating_open_ai_endpoint = True` in config.py

In [ ]:
if config.get('is_creating_open_ai_endpoint', False):
    print("=" * 60)
    print("DEPLOYING: Azure OpenAI Service")
    print("=" * 60)
    print(f"Resource Name: {config['open_ai_resource_name']}")
    print(f"Location: {config['location']}")
    print(f"Reasoning Model: {config['open_ai_reasoning_model_model_name']}")
    print(f"Embedding Model: {config['open_ai_embedding_model_name']}")
    
    # Check if resource exists and handle location mismatch
    should_deploy, effective_location, message = check_and_prepare_deployment(
        resource_name=config['open_ai_resource_name'],
        resource_type="openai",
        config_location=config['location'],
        subscription_id=config['subscription_id'],
        resource_group=config.get('open_ai_resource_group', '') or config['resource_group'],
        credential=credential
    )
    print(f"\n{message}")
    
    if should_deploy:
        # Load parameters from template (resolved from config.py)
        aoai_params = get_parameters_for_deployment("AOAI", config)
        
        # Deploy
        aoai_outputs = deploy_arm(
            template_file_path="templates/AOAI.Template.json",
            deployment_name="deepwiki-aoai",
            parameters=aoai_params,
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group']
        )
        
        print(f"\n✓ Azure OpenAI deployed successfully")
    else:
        print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
        print(f"  Please resolve the issue above before continuing.")
else:
    print("=" * 60)
    print("SKIPPING: Azure OpenAI Service")
    print("=" * 60)
    print(f"Reason: is_creating_open_ai_endpoint = False in config.py")
    print(f"Using existing Azure OpenAI resource: {config['open_ai_resource_name']}")

## Block 6: Deploy Application Insights & Log Analytics

Deploy Application Insights with Log Analytics workspace for monitoring.

In [ ]:
print("=" * 60)
print("DEPLOYING: Application Insights & Log Analytics")
print("=" * 60)
print(f"Application Insights: {config['application_insights_name']}")
print(f"Log Analytics: {config['log_analytics_workspace_name']}")
print(f"Location: {config['location']}")

# Check if Application Insights exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['application_insights_name'],
    resource_type="appinsights",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    appinsights_params = get_parameters_for_deployment("APPINSIGHT", config)
    
    # Deploy
    appinsights_outputs = deploy_arm(
        template_file_path="templates/APPINSIGHT.Template.json",
        deployment_name="deepwiki-appinsights",
        parameters=appinsights_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    # Store connection string for App Service deployment
    appinsights_connection_string = appinsights_outputs.get('applicationInsightsConnectionString', {}).get('value', '')
    
    print(f"\n✓ Application Insights deployed successfully")
    if appinsights_connection_string:
        print(f"  - Connection String: {appinsights_connection_string[:50]}...")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 6.5: Deploy Azure Container Registry

Deploy Azure Container Registry for storing the DeepWiki container image.
The Managed Identity will be granted AcrPull role for pulling images.

In [ ]:
print("=" * 60)
print("DEPLOYING: Azure Container Registry")
print("=" * 60)
print(f"Container Registry: {config['container_registry_name']}")
print(f"Location: {config['location']}")

# Check if ACR exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['container_registry_name'],
    resource_type="acr",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    acr_params = get_parameters_for_deployment("ACR", config)
    
    # Deploy
    acr_outputs = deploy_arm(
        template_file_path="templates/ACR.Template.json",
        deployment_name="deepwiki-acr",
        parameters=acr_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    print(f"\n✓ Container Registry deployed successfully")
    if acr_outputs:
        print(f"  - Login Server: {acr_outputs.get('containerRegistryLoginServer', {}).get('value', 'N/A')}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 7: Deploy Azure App Service

Deploy App Service (Linux container mode) with container from Azure Container Registry.

**Container Configuration:**
- Uses custom Docker image from ACR (built via `publish-web.ps1`)
- Managed Identity for ACR authentication
- Image: `{container_registry_name}.azurecr.io/{container_image_name}:latest`

**Network Restrictions Applied:**
- Allow: AzureTrafficManager (priority 200)
- Allow: CorpNetPublic (priority 400)
- Allow: CorpNetSAW (priority 401)
- Deny: All other traffic

In [ ]:
print("=" * 60)
print("DEPLOYING: Azure App Service")
print("=" * 60)
print(f"App Service Plan: {config['app_service_plan_name']}")
print(f"App Service: {config['app_service_name']}")
print(f"SKU: {config['app_service_plan_sku']}")
print(f"Container Registry: {config['container_registry_name']}")
print(f"Container Image: {config['container_image_name']}:latest")
print(f"Network Restrictions: {config['enable_app_service_network_restrictions']}")

# Check if App Service exists and handle location mismatch
should_deploy, effective_location, message = check_and_prepare_deployment(
    resource_name=config['app_service_name'],
    resource_type="app_service",
    config_location=config['location'],
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    credential=credential
)
print(f"\n{message}")

if should_deploy:
    # Load parameters from template (resolved from config.py)
    appservice_params = get_parameters_for_deployment("WEB", config)
    
    # Override applicationInsightsConnectionString if we have it from previous deployment
    if 'appinsights_connection_string' in globals() and appinsights_connection_string:
        appservice_params['applicationInsightsConnectionString'] = appinsights_connection_string
    
    # Deploy
    appservice_outputs = deploy_arm(
        template_file_path="templates/WEB.Template.json",
        deployment_name="deepwiki-appservice",
        parameters=appservice_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    
    print(f"\n✓ App Service deployed successfully")
    if appservice_outputs:
        print(f"  - URL: {appservice_outputs.get('appServiceUrl', {}).get('value', 'N/A')}")
        print(f"  - Container: {appservice_outputs.get('containerImageFull', {}).get('value', 'N/A')}")
else:
    print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
    print(f"  Please resolve the issue above before continuing.")

## Block 8: Configure Role Assignments

Assign RBAC roles to the Managed Identity:
- **Cognitive Services OpenAI User** on Azure OpenAI
- **Storage Blob Data Contributor** on Storage Account
- **Monitoring Metrics Publisher** on Application Insights

In [ ]:
print("=" * 60)
print("DEPLOYING: Role Assignments")
print("=" * 60)

# Ensure we have Azure credential (in case kernel was restarted)
from azure.identity import AzureCliCredential
from azure.mgmt.msi import ManagedServiceIdentityClient

try:
    credential
except NameError:
    credential = AzureCliCredential()
    print("✓ Azure credential initialized")

# Get managed identity principal ID from previous deployment or retrieve from Azure
_has_principal_id = 'managed_identity_principal_id' in globals() and globals().get('managed_identity_principal_id')

if not _has_principal_id:
    print("Retrieving Managed Identity info from Azure...")
    msi_client = ManagedServiceIdentityClient(credential, config['subscription_id'])
    identity = msi_client.user_assigned_identities.get(
        config['resource_group'],
        config['dri_copilot_identity_name']
    )
    managed_identity_principal_id = identity.principal_id
    managed_identity_client_id = identity.client_id
    print(f"  - Principal ID: {managed_identity_principal_id}")
    print(f"  - Client ID: {managed_identity_client_id}")
else:
    print(f"Using Managed Identity from previous deployment:")
    print(f"  - Principal ID: {managed_identity_principal_id}")

# Determine Azure OpenAI resource group (may be different from main resource group)
open_ai_rg = config.get('open_ai_resource_group', '') or ''
if open_ai_rg:
    print(f"\n📍 Azure OpenAI in external resource group: {open_ai_rg}")
else:
    print(f"\n📍 Azure OpenAI in same resource group: {config['resource_group']}")

# Load parameters from template (resolved from config.py)
role_params = get_parameters_for_deployment("RBAC", config)

# Override managedIdentityPrincipalId with the actual value from deployment
role_params['managedIdentityPrincipalId'] = managed_identity_principal_id

# Clear deploymentIdentityPrincipalId to avoid org policy violation (SFI blocks User RBAC)
role_params['deploymentIdentityPrincipalId'] = ""

print(f"\nAssigning roles to Managed Identity: {config['dri_copilot_identity_name']}")
print(f"  - Azure OpenAI ({config['open_ai_resource_name']}): Cognitive Services OpenAI User")
print(f"  - Storage ({config['storage_account_name']}): Storage Blob Data Contributor")
print(f"  - App Insights ({config['application_insights_name']}): Monitoring Metrics Publisher")
print(f"\n⚠ Note: User role assignments skipped (blocked by org SFI policy)")

# Deploy role assignments
role_outputs = deploy_arm(
    template_file_path="templates/RBAC.Template.json",
    deployment_name="deepwiki-role-assignments",
    parameters=role_params,
    subscription_id=config['subscription_id'],
    resource_group=config['resource_group'],
    skip_role_assignment=False
)

print(f"\n✓ Role assignments completed successfully")

## Block 9: Deploy Network Security Perimeter

Deploy NSP with enforced mode to protect Storage Account.

**NSP Rules (Hardcoded - not configurable):**
- Inbound: Allow traffic from current subscription
- Inbound: Allow MicrosoftPublicIPSpace service tag
- Outbound: Allow all FQDNs (*)

**Note:** Only run this block if NSP is configured in config.py

In [ ]:
nsp_name = config.get('nsp_name', '')
nsp_profile_name = config.get('nsp_profile_name', '')
nsp_access_mode_storage = config.get('nsp_access_mode_for_storage', '')

if nsp_name and not nsp_name.startswith('__'):
    print("=" * 60)
    print("DEPLOYING: Network Security Perimeter")
    print("=" * 60)
    print(f"NSP Name: {nsp_name}")
    print(f"NSP Profile: {nsp_profile_name}")
    print(f"Access Mode (Storage): {nsp_access_mode_storage or 'Not configured'}")
    
    # Check if NSP exists and handle location mismatch
    should_deploy, effective_location, message = check_and_prepare_deployment(
        resource_name=nsp_name,
        resource_type="nsp",
        config_location=config['location'],
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        credential=credential
    )
    print(f"\n{message}")
    
    if should_deploy:
        # Load parameters from template (resolved from config.py)
        nsp_params = get_parameters_for_deployment("NSP", config)
        
        # Deploy NSP
        nsp_outputs = deploy_arm(
            template_file_path="templates/NSP.Template.json",
            deployment_name="deepwiki-nsp",
            parameters=nsp_params,
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group']
        )
        
        print(f"\n✓ NSP deployed successfully")
        
        # Associate NSP with Storage Account (only if access mode is configured)
        if nsp_access_mode_storage and nsp_access_mode_storage in ['Enforced', 'Learning']:
            print(f"\nAssociating NSP with Storage Account (Mode: {nsp_access_mode_storage})...")
            
            nsp_assoc_params = {
                "nspName": nsp_name,
                "nspProfileName": nsp_profile_name,
                "nspAccessMode": nsp_access_mode_storage,
                "resourceName": config['storage_account_name'],
                "resourceType": "storage"
            }
            
            nsp_assoc_outputs = deploy_arm(
                template_file_path="templates/NSP.Associations.Template.json",
                deployment_name="deepwiki-nsp-storage-assoc",
                parameters=nsp_assoc_params,
                subscription_id=config['subscription_id'],
                resource_group=config['resource_group']
            )
            
            print(f"✓ NSP associated with Storage Account (Mode: {nsp_access_mode_storage})")
        else:
            print(f"\n⚠ Skipping Storage Account NSP association")
            print(f"  Reason: nsp_access_mode_for_storage not set to 'Enforced' or 'Learning'")
    else:
        print(f"\n⚠ Deployment SKIPPED due to location mismatch.")
        print(f"  Please resolve the issue above before continuing.")
else:
    print("=" * 60)
    print("SKIPPING: Network Security Perimeter")
    print("=" * 60)
    print("Reason: NSP not configured in config.py")
    print("To enable NSP, set 'nsp_name' and 'nsp_profile_name' in config.py")

## Block 10: Deploy Key Vault

Deploy Azure Key Vault with RBAC authorization. Required by the AML workspace.

The Key Vault is created with:
- RBAC authorization (no access policies)  
- Soft delete enabled (90 days)
- Purge protection enabled  
- Public network access disabled (Azure Services bypass)


In [ ]:
# Deploy Key Vault
print("=" * 60)
print("DEPLOYING: Azure Key Vault")
print("=" * 60)
print(f"Key Vault Name:  {config['key_vault_name']}")
print(f"Location:        {config['location']}")
print(f"RBAC Enabled:    {config.get('key_vault_enable_rbac', True)}")

# Check if already exists
should_deploy, effective_location, message = check_and_prepare_deployment(
    config['key_vault_name'],
    "Microsoft.KeyVault/vaults",
    config['location'],
    config['subscription_id'],
    config['resource_group'],
    credential
)
print(f"\n{message}")

if should_deploy:
    # Resolve parameters from template
    params = get_parameters_for_deployment("KV", config)

    outputs = deploy_arm(
        template_file_path="templates/KV.Template.json",
        deployment_name=f"deepwiki-kv-{config['location']}",
        parameters=params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )

    print(f"\n✓ Key Vault deployment completed")
    if outputs:
        print(f"  URI: {outputs.get('keyVaultUri', {}).get('value', 'N/A')}")
else:
    print("Deployment skipped — see message above.")


## Block 10a: Grant Key Vault RBAC Permissions

Grant required Key Vault roles to the managed identity and the deployer (per `permission.md`):

| Principal | Role |
|-----------|------|
| Managed identity (`mid-orcas-deepwiki`) | Key Vault Crypto User |
| Managed identity (`mid-orcas-deepwiki`) | Key Vault Secrets Officer |
| Managed identity (`mid-orcas-deepwiki`) | Key Vault Certificate User |
| Deployer (you) | Key Vault Administrator |


In [ ]:
# Grant Key Vault RBAC permissions using the RBAC ARM template
print("=" * 60)
print("GRANTING: Key Vault RBAC Permissions")
print("=" * 60)

# Get managed identity principal ID
from azure.mgmt.msi import ManagedServiceIdentityClient
msi_client = ManagedServiceIdentityClient(credential, config['subscription_id'])
identity = msi_client.user_assigned_identities.get(
    config['resource_group'], config['dri_copilot_identity_name']
)
managed_identity_principal_id = identity.principal_id
print(f"Managed Identity Principal ID: {managed_identity_principal_id}")

# Use RBAC template to assign KV roles
rbac_params = {
    "managedIdentityPrincipalId": managed_identity_principal_id,
    "deploymentIdentityPrincipalId": config['deployment_identity_principal_id'],
    "deploymentIdentityPrincipalType": "User",
    "keyVaultName": config['key_vault_name'],
    # Only assign KV roles — leave others empty to skip
    "openAiResourceName": "",
    "openAiResourceGroup": "",
    "storageAccountName": "",
    "applicationInsightsName": "",
    "searchServiceName": "",
    "machineLearningWorkspaceName": ""
}

try:
    outputs = deploy_arm(
        template_file_path="templates/RBAC.Template.json",
        deployment_name=f"deepwiki-rbac-kv-{config['location']}",
        parameters=rbac_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group']
    )
    print(f"\n✓ Key Vault RBAC permissions granted successfully")
    print(f"  Managed identity roles: Crypto User, Secrets Officer, Certificate User")
    print(f"  Deployer role: Key Vault Administrator")
except Exception as e:
    if "RoleAssignmentExists" in str(e):
        print(f"\n✓ Role assignments already exist (this is normal)")
    else:
        print(f"\n✗ RBAC assignment failed: {e}")
        raise


## Block 10b: Associate Key Vault with NSP

Associate the Key Vault with the existing Network Security Perimeter deployed in `deploy_required.ipynb`.

Since Key Vault has `publicNetworkAccess: "Disabled"`, NSP association is required for Azure services to reach it through the perimeter.

The access mode is configured via `nsp_access_mode_for_key_vault` in `config.py` (default: `"Enforced"`).


In [ ]:
# Associate Key Vault with NSP
nsp_name = config.get('nsp_name', '')
nsp_profile_name = config.get('nsp_profile_name', '')
nsp_access_mode_kv = config.get('nsp_access_mode_for_key_vault', '')

if nsp_name and not nsp_name.startswith('__') and nsp_access_mode_kv in ['Enforced', 'Learning']:
    print("=" * 60)
    print("ASSOCIATING: Key Vault with NSP")
    print("=" * 60)
    print(f"NSP Name:      {nsp_name}")
    print(f"NSP Profile:   {nsp_profile_name}")
    print(f"Key Vault:     {config['key_vault_name']}")
    print(f"Access Mode:   {nsp_access_mode_kv}")

    nsp_assoc_params = {
        "nspName": nsp_name,
        "nspProfileName": nsp_profile_name,
        "nspAccessMode": nsp_access_mode_kv,
        "resourceName": config['key_vault_name'],
        "resourceType": "keyvault"
    }

    try:
        nsp_assoc_outputs = deploy_arm(
            template_file_path="templates/NSP.Associations.Template.json",
            deployment_name="deepwiki-nsp-kv-assoc",
            parameters=nsp_assoc_params,
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group']
        )
        print(f"\n✓ Key Vault associated with NSP (Mode: {nsp_access_mode_kv})")
    except Exception as e:
        print(f"\n✗ NSP association failed: {e}")
        raise
else:
    print("SKIPPING: Key Vault NSP Association")
    if not nsp_name or nsp_name.startswith('__'):
        print("  Reason: NSP not configured in config.py")
    else:
        print(f"  Reason: nsp_access_mode_for_key_vault = '{nsp_access_mode_kv}' (must be 'Enforced' or 'Learning')")


## Block 11: Deploy Azure Machine Learning Workspace

Deploy AML workspace linked to **existing** resources (no duplicates created):

| Component | Reuses Existing | Falls back to prefix |
|-----------|----------------|---------------------|
| Storage Account | `bloborcasdeepwiki` (from deploy_required) | `{prefix}storage` only if not configured |
| Key Vault | `kvorcascodewiki` (from Block 2) | `{prefix}keyvault` only if not configured |
| Application Insights | `appinsite-orcas-deepwiki-ea` (from deploy_required) | `{prefix}appinsights` only if not configured |
| Log Analytics | `log-orcas-deepwiki-ea` (from deploy_required) | `{prefix}loganalytics` only if not configured |
| Container Registry | Always created | `cr{prefix}` (AML-specific, no existing equivalent) |

The ARM template uses conditions to skip creating resources when `existing*Name` parameters are provided (non-empty). The prefix is only used as fallback for components NOT configured in `config.py`.

**Note:** This takes 10-15 minutes.


In [ ]:
# Deploy Azure Machine Learning Workspace
print("=" * 60)
print("DEPLOYING: Azure Machine Learning Workspace")
print("=" * 60)
print(f"Workspace Name:   {config['machine_learning_workspace_name']}")
print(f"Location:         {config['location']}")
print(f"Managed Identity: {config['dri_copilot_identity_name']}")
print(f"OpenAI Resource:  {config['open_ai_resource_name']}")
print(f"VNet enabled:     {config['is_creating_vnet_for_azure_ml']}")

# Show which resources are reused vs created
prefix = config['machine_learning_workspace_sub_components_name_prefix']
reuse_map = {
    "Storage Account": (config.get('storage_account_name', ''), f"{prefix}storage"),
    "Key Vault":       (config.get('key_vault_name', ''),       f"{prefix}keyvault"),
    "App Insights":    (config.get('application_insights_name', ''), f"{prefix}appinsights"),
    "Log Analytics":   (config.get('log_analytics_workspace_name', ''), f"{prefix}loganalytics"),
}
print(f"\nResource resolution (existing → reuse, empty → create from prefix '{prefix}'):")
for label, (existing, fallback) in reuse_map.items():
    if existing:
        print(f"  ✓ {label:20s} → reuse existing: {existing}")
    else:
        print(f"  + {label:20s} → create new: {fallback}")
print(f"  + {'Container Registry':20s} → always create: cr{prefix}")

# Resolve parameters from template
params = get_parameters_for_deployment("AML", config)

# NOTE: skip_role_assignment=True because:
# 1. MCAPS SFI policy blocks persistent RBAC for principalType="User"
#    (Key Vault Crypto User, Key Vault Administrator, Contributor)
# 2. Managed identity roles (ServicePrincipal) are already assigned via deploy_required
# 3. User roles (Key Vault Admin, Contributor) must be assigned via PIM instead
print(f"\nStarting deployment (skip_role_assignment=True due to MCAPS SFI policy)...")
print("  ⚠ User-level roles (KV Admin, Contributor) must be assigned via PIM")
print("  ✓ Managed identity roles already assigned via deploy_required")

try:
    outputs = deploy_arm(
        template_file_path="templates/AML.Template.json",
        deployment_name=f"deepwiki-aml-{config['location']}",
        parameters=params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        skip_role_assignment=True
    )
    print(f"\n✓ Azure ML deployment completed successfully")
    print(f"  Workspace: {config['machine_learning_workspace_name']}")
except Exception as e:
    print(f"\n✗ Azure ML deployment failed: {e}")
    raise


## Block 11a: Update AML Datastores to Managed Identity

Switch default AML datastores from account key auth to managed identity auth.


In [ ]:
# Update AML datastores to use managed identity authentication
try:
    set_datastore_credential_to_managed_identity(
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        workspace_name=config['machine_learning_workspace_name']
    )
except Exception as e:
    print(f"\n⚠ Datastore update encountered issues: {e}")
    print("  You can update datastores manually from Azure Portal if needed.")


## Block 11b: Provision AML Managed Network (if VNet enabled)

Activate the managed network outbound rules. Required when `is_creating_vnet_for_azure_ml = True`.

The managed identity needs "Azure AI Enterprise Network Connection Approver" on the storage account (granted in deploy_required).


In [ ]:
# Provision AML Managed Network
if config.get('is_creating_vnet_for_azure_ml', False):
    print("Provisioning managed network for AML workspace...")
    print("Note: If you see a permissions error, wait 2-3 minutes and retry.")
    try:
        provision_managed_network(
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group'],
            workspace_name=config['machine_learning_workspace_name'],
            include_spark=False
        )
    except Exception as e:
        print(f"\n⚠ Managed network provisioning issue: {e}")
        print("  You can provision it from Azure Portal if needed.")
else:
    print("SKIPPING: AML Managed Network Provisioning")
    print(f"Reason: is_creating_vnet_for_azure_ml = False in config.py")


## Block 12: Deploy Azure AI Search Service

Deploy Azure AI Search linked to the **existing** storage account (`bloborcasdeepwiki`).

The template:
- Creates the Search service with system-assigned identity
- Grants the Search service's system identity **Storage Blob Data Reader** on the storage account
- Grants the managed identity **Search Service Contributor** and **Search Index Data Reader**
- Grants the deployer **Search Service Contributor** and **Search Index Data Reader**
- Grants the deployer **Contributor** on the resource group


In [ ]:
# Deploy Azure AI Search Service
print("=" * 60)
print("DEPLOYING: Azure AI Search Service")
print("=" * 60)
print(f"Search Service:   {config['search_service_name']}")
print(f"Storage Account:  {config['storage_account_name']}")
print(f"Location:         {config['location']}")

# Resolve parameters from template (uses ACS.Parameters.json + config.py)
params = get_parameters_for_deployment("ACS", config)

# NOTE: skip_role_assignment=True because MCAPS SFI policy blocks persistent
# RBAC for principalType="User" (Search Service Contributor, Index Data Reader, Contributor).
# The managed identity + Search system identity roles (ServicePrincipal) are NOT in the
# roleAssignment resources — they're handled separately. But the template mixes User and
# ServicePrincipal assignments together, so we must skip all and handle SP roles separately.
print(f"\nStarting deployment (skip_role_assignment=True due to MCAPS SFI policy)...")
print("  ⚠ User-level roles (Search Contributor, Index Data Reader) must be assigned via PIM")
print("  ✓ Managed identity roles already assigned via deploy_required / permission.md")

try:
    outputs = deploy_arm(
        template_file_path="templates/ACS.Template.json",
        deployment_name=f"deepwiki-acs-{config['location']}",
        parameters=params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        skip_role_assignment=True
    )
    print(f"\n✓ Azure AI Search deployment completed successfully")
    print(f"  Endpoint: https://{config['search_service_name']}.search.windows.net")
    print(f"\n⚠ IMPORTANT: Role assignments were skipped. You need to manually assign:")
    print(f"  1. Search system identity → Storage Blob Data Reader on {config['storage_account_name']}")
    print(f"  2. Managed identity → Search Service Contributor + Search Index Data Reader on {config['search_service_name']}")
    print(f"  These can be done via Azure Portal or 'az role assignment create' commands.")
except Exception as e:
    print(f"\n✗ Azure AI Search deployment failed: {e}")
    raise


## Block 13: Deploy Private Network (VNet + Private Endpoints)

Runs only when `private_network = True` in `config.py`. This block:

1. Deploys `NETWORK.Template.json` — a virtual network (`pe-subnet` + delegated `app-service-subnet`), the `privatelink.openai.azure.com` and `privatelink.search.windows.net` private DNS zones, private endpoints for OpenAI and AI Search, App Service regional VNet integration, and the **Azure AI Enterprise Network Connection Approver** role for the managed identity on OpenAI + Search.
2. Disables public network access on the existing OpenAI account (with outbound DLP — required by the `CloudGov_DLP_AzOpenAI` policy). For greenfield deploys (`is_creating_open_ai_endpoint = True`) the AOAI template handles this instead.
3. Adds AML managed-network outbound private endpoints to OpenAI + Search and provisions the managed network, so the processor keeps working after public access is disabled.

AI Search public access is set declaratively by the ACS template via its `privateNetwork` parameter. If you hit a `400 permissions` error in step 3, wait 2–3 minutes for RBAC to propagate and re-run this cell.


In [ ]:
# Block 13: Deploy Private Network (VNet + Private Endpoints)
# Runs only when private_network = True in config.py. Idempotent: re-running
# updates the existing resources in place.
from utils import (
    get_parameters_for_deployment, deploy_arm,
    provision_managed_network,
    add_aml_managed_private_endpoint_rule, set_openai_private_network,
)

if config.get('private_network', True):
    print("=" * 60)
    print("DEPLOYING: Private Network (VNet, Private Endpoints, DNS)")
    print("=" * 60)
    print(f"VNet:            {config['vnet_name']} ({config['vnet_address_prefix']})")
    print(f"PE subnet:       {config['private_endpoint_subnet_name']} ({config['private_endpoint_subnet_prefix']})")
    print(f"App subnet:      {config['app_service_subnet_name']} ({config['app_service_subnet_prefix']})")
    print(f"Private targets: {config['open_ai_resource_name']}, {config['search_service_name']}")

    # 1) VNet + subnets + private DNS zones + private endpoints (OpenAI, Search)
    #    + App Service regional VNet integration
    #    + 'Azure AI Enterprise Network Connection Approver' role for the managed
    #      identity on OpenAI and Search (needed to approve the AML managed PEs).
    network_params = get_parameters_for_deployment("NETWORK", config)
    deploy_arm(
        template_file_path="templates/NETWORK.Template.json",
        deployment_name=f"deepwiki-network-{config['location']}",
        parameters=network_params,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
    )
    print("\n\u2713 Private network resources deployed (VNet, PEs, DNS, App Service integration)")

    # 2) Lock down the OpenAI account. The AOAI template applies this for
    #    greenfield deploys, but it is SKIPPED when is_creating_open_ai_endpoint
    #    = False, so apply it to the existing account here. publicNetworkAccess=
    #    Disabled REQUIRES restrictOutboundNetworkAccess=true (CloudGov DLP).
    if not config.get('is_creating_open_ai_endpoint', False):
        print("\nDisabling public network access on existing OpenAI account...")
        try:
            set_openai_private_network(
                subscription_id=config['subscription_id'],
                resource_group=config.get('open_ai_resource_group') or config['resource_group'],
                openai_resource_name=config['open_ai_resource_name'],
                enabled=True,
            )
            print("\u2713 OpenAI: publicNetworkAccess=Disabled, outbound DLP enabled")
        except Exception as e:
            print(f"\u26a0 Could not update OpenAI network settings: {e}")

    # 3) Give the AML managed network its own private path to OpenAI + Search so
    #    the processor keeps working after public access is off. Needs the
    #    approver role from step 1 - on a 400 'permissions' error, wait 2-3 min
    #    for RBAC to propagate and re-run this cell.
    if config.get('is_creating_vnet_for_azure_ml', False):
        sub = config['subscription_id']
        rg = config['resource_group']
        aoai_rg = config.get('open_ai_resource_group') or rg
        targets = [
            ("pe-aoai-deepwiki",
             f"/subscriptions/{sub}/resourceGroups/{aoai_rg}/providers/Microsoft.CognitiveServices/accounts/{config['open_ai_resource_name']}",
             "account"),
            ("pe-acs-deepwiki",
             f"/subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.Search/searchServices/{config['search_service_name']}",
             "searchService"),
        ]
        for rule_name, target_id, sub_target in targets:
            try:
                add_aml_managed_private_endpoint_rule(
                    subscription_id=sub,
                    resource_group=rg,
                    workspace_name=config['machine_learning_workspace_name'],
                    rule_name=rule_name,
                    target_resource_id=target_id,
                    subresource_target=sub_target,
                )
                print(f"\u2713 AML managed PE rule added: {rule_name}")
            except Exception as e:
                print(f"\u26a0 AML managed PE rule '{rule_name}' failed (RBAC may still be propagating, retry in 2-3 min): {e}")

        provision_managed_network(
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group'],
            workspace_name=config['machine_learning_workspace_name'],
            include_spark=False,
        )

    print("\n\u2713 Private network configuration complete")
    print("  OpenAI + AI Search are now reachable only through private endpoints.")
else:
    print("=" * 60)
    print("SKIPPING: Private Network")
    print("=" * 60)
    print("Reason: private_network = False in config.py")
    print("Set private_network = True to deploy the VNet, private endpoints and lock down OpenAI/Search.")


## Deployment Summary

Verify all resources and permissions are in place.


In [ ]:
# Deployment Summary
print("=" * 60)
print("DEPLOYMENT SUMMARY")
print("=" * 60)

# Check each resource
resources_to_check = [
    ("Key Vault",    config['key_vault_name'],               "Microsoft.KeyVault/vaults"),
    ("AI Search",    config['search_service_name'],           "Microsoft.Search/searchServices"),
]

print("\n--- Planning Resources ---")
for label, name, rtype in resources_to_check:
    exists, info = check_resource_exists(
        name, rtype, config['subscription_id'], config['resource_group'], credential
    )
    status = f"✓ {info['location']}" if exists else "✗ NOT FOUND"
    print(f"  {label:20s} {name:30s} {status}")

# Check AML
try:
    from azure.mgmt.machinelearningservices import AzureMachineLearningWorkspaces
    aml_client = AzureMachineLearningWorkspaces(credential, config['subscription_id'])
    ws = aml_client.workspaces.get(config['resource_group'], config['machine_learning_workspace_name'])
    print(f"  {'AML Workspace':20s} {config['machine_learning_workspace_name']:30s} ✓ {ws.location}")
except Exception:
    print(f"  {'AML Workspace':20s} {config['machine_learning_workspace_name']:30s} ✗ NOT FOUND")

print("\n--- Permission Checklist (per permission.md) ---")
print("  Key Vault:")
print("    ✓ Managed identity: Crypto User, Secrets Officer, Certificate User (via RBAC template)")
print("    ✓ Deployer: Key Vault Administrator (via RBAC template)")
print("    ✓ NSP association (if configured)")
print("  AI Search:")
print("    ✓ Managed identity: Search Index Data Reader, Search Service Contributor (via ACS template)")
print("    ✓ Search system identity: Storage Blob Data Reader on storage (via ACS template)")
print("    ✓ Data source configured (deepwiki-storage-datasource)")
print("  AML Workspace:")
print("    ✓ Managed identity: AzureML Data Scientist (via AML template)")
print("    ✓ Deployer: Contributor (via AML template)")
print("  Storage (from deploy_required):")
print("    ✓ Managed identity: Blob Data Contributor, File Data Privileged Contributor")
print("  OpenAI (from deploy_required):")
print("    ✓ Managed identity: Cognitive Services User, Cognitive Services OpenAI User")

print(f"\n{'=' * 60}")
print("✓ All planning infrastructure deployed and configured")
print("  Note: Search index will be created later when the search feature is implemented.")
print(f"{'=' * 60}")
